# GPT-OSS 20B Fine-Tuning with Unsloth

This notebook demonstrates fine-tuning the OpenAI GPT-OSS 20B model using:
- **Unsloth**: For efficient training with reduced memory usage
- **LoRA**: Parameter-efficient fine-tuning (train only ~1% of parameters)
- **Dataset**: Stanford Alpaca - 52K instruction-following examples

## Prerequisites
- CUDA-compatible GPU with at least 12GB VRAM
- Python 3.8+

## Step 1: Install Dependencies

In [1]:
%%capture
import os, importlib.util

# Install uv package manager
!pip install --upgrade -qqq uv

# Install core dependencies
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil = "pillow"
    
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

# Upgrade to latest versions
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

## Step 2: Load Model and Tokenizer

We load the GPT-OSS 20B model with:
- **4-bit quantization**: Reduces memory usage by ~75%
- **Max sequence length**: 1024 tokens (adjustable)
- **Auto dtype detection**: Automatically selects optimal precision

In [2]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 1024  # Can be increased for longer contexts
dtype = None  # Auto-detect (typically bfloat16 or float16)
load_in_4bit = True  # Enable 4-bit quantization for memory efficiency

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    full_finetuning=False,  # Use LoRA instead of full fine-tuning
    # token="YOUR_HF_TOKEN",  # Uncomment if model requires authentication
)

print(f"Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

## Step 3: Configure LoRA Adapters

LoRA (Low-Rank Adaptation) enables efficient fine-tuning by:
- Training only adapter layers instead of the entire model
- Reducing trainable parameters to ~1% of total parameters
- Maintaining performance while drastically reducing memory requirements

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # LoRA rank (higher = more capacity, try 8, 16, 32, 64)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",    # MLP layers
    ],
    lora_alpha=16,  # LoRA scaling factor (typically 2*r)
    lora_dropout=0,  # Dropout for LoRA layers (0 is optimized)
    bias="none",  # Bias training ("none" is optimized)
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=3407,  # For reproducibility
    use_rslora=False,  # Rank-stabilized LoRA
    loftq_config=None,  # LoftQ quantization config
)

print("LoRA adapters configured successfully!")

## Step 4: Load and Prepare Dataset

We use the **tatsu-lab/alpaca** dataset, a popular open-source instruction-following dataset containing:
- 52K instruction-following examples
- Diverse tasks: question answering, creative writing, reasoning, code generation, etc.
- Simple format: instruction, optional input, and output

**Alternative datasets you can use:**
- `databricks/databricks-dolly-15k` - 15K high-quality human-generated examples
- `Open-Orca/OpenOrca` - Large-scale instruction dataset  
- `mlabonne/guanaco-llama2-1k` - Smaller dataset for quick testing
- Your own custom dataset in similar format

In [ ]:
from datasets import load_dataset

# Load Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train")

# Optional: Use a subset for faster testing
# dataset = dataset.select(range(1000))  # Use only first 1000 examples

print(f"Dataset loaded: {len(dataset)} examples")
print(f"Features: {dataset.features}")

# Display sample
print("\nSample data:")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Input: {dataset[0]['input']}")
print(f"Output: {dataset[0]['output'][:200]}...")  # First 200 chars

### Format Dataset for Training

In [ ]:
def formatting_prompts_func(examples):
    """
    Convert Alpaca format to GPT-OSS chat format.
    
    Alpaca format: {instruction, input, output}
    GPT-OSS format: Structured conversation with system, user, assistant messages
    """
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Combine instruction and input if input exists
        if input_text.strip():
            user_message = f"{instruction}\n\nInput: {input_text}"
        else:
            user_message = instruction
        
        # Create conversation in chat format
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output}
        ]
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    
    return {"text": texts}

# Format dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset formatted successfully!")
print(f"\nFormatted example (first 800 chars):")
print(dataset[0]['text'][:800] + "...")

## Step 5: Configure Training

Training configuration with:
- **Supervised Fine-Tuning (SFT)**: Standard fine-tuning approach
- **Response-only training**: Train only on assistant responses, ignore user inputs
- **8-bit AdamW optimizer**: Memory-efficient optimizer

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# Define GPT-OSS specific training parameters
gpt_oss_kwargs = {
    "instruction_part": "<|start|>user<|message|>",
    "response_part": "<|start|>assistant<|channel|>final<|message|>"
}

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,  # Batch size per GPU
        gradient_accumulation_steps=4,  # Effective batch size = 1 * 4 = 4
        warmup_steps=5,  # Learning rate warmup
        max_steps=60,  # Number of training steps (set to None for full training)
        # num_train_epochs=1,  # Uncomment for full epoch training
        learning_rate=2e-4,  # Learning rate
        logging_steps=1,  # Log every step
        optim="adamw_8bit",  # 8-bit optimizer for memory efficiency
        weight_decay=0.001,  # L2 regularization
        lr_scheduler_type="linear",  # Learning rate scheduler
        seed=3407,  # Random seed for reproducibility
        output_dir="outputs",  # Output directory for checkpoints
        report_to="none",  # Logging platform (use "wandb", "tensorboard", etc.)
    ),
)

# Configure training to focus only on assistant responses
trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

print("Trainer configured successfully!")
print(f"Training on {len(trainer.train_dataset)} samples")

### Verify Training Masking

Check that instruction parts are masked (not trained on) and only responses are trained.

In [ ]:
# Display full input
print("Full input example:")
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"])[:500] + "...")

# Display masked labels (only responses are visible, rest is padding)
print("\n" + "="*80)
print("Masked labels (what the model trains on):")
masked_output = tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x 
    for x in trainer.train_dataset[0]["labels"]
]).replace(tokenizer.pad_token, " ")
print(masked_output[:500] + "...")

## Step 6: Train the Model

Begin training with the configured parameters.
To resume from a checkpoint, use: `trainer.train(resume_from_checkpoint=True)`

In [ ]:
# Display initial memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Memory reserved before training: {start_gpu_memory} GB")
print("\nStarting training...\n")

# Train the model
trainer_stats = trainer.train()

# Display training statistics
print("\n" + "="*80)
print("Training completed!")
print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")

# Display memory usage
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"Peak memory: {used_memory} GB ({used_percentage}% of total)")
print(f"Memory used for training: {used_memory_for_lora} GB")

## Step 7: Test the Fine-Tuned Model

Run inference to verify the model has learned from the training data.

GPT-OSS supports three reasoning effort levels:
- **low**: Fast responses, minimal reasoning (best for direct instructions)
- **medium**: Balanced performance and speed
- **high**: Maximum reasoning capability, slower responses

In [ ]:
from transformers import TextStreamer

# Test with a generic instruction similar to training data
messages = [
    {
        "role": "user",
        "content": "Give three tips for staying healthy."
    },
]

# Format input
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="low",  # Use "low" for direct answers, "medium/high" for reasoning
).to("cuda")

# Generate response
print("User: Give three tips for staying healthy.\n")
print("Assistant: ", end="")
_ = model.generate(
    **inputs,
    max_new_tokens=256,  # Increase for longer responses
    streamer=TextStreamer(tokenizer, skip_prompt=True)
)

### Test with Another Example

In [ ]:
# Test with a more complex instruction
messages = [
    {
        "role": "user",
        "content": "Explain the concept of photosynthesis in simple terms."
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="low",
).to("cuda")

print("User: Explain the concept of photosynthesis in simple terms.\n")
print("Assistant: ", end="")
_ = model.generate(
    **inputs,
    max_new_tokens=300,
    streamer=TextStreamer(tokenizer, skip_prompt=True)
)

## Step 8: Save the Fine-Tuned Model

Save the LoRA adapters for later use.

In [ ]:
# Save locally
output_dir = "gpt_oss_lora_adapters"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to: {output_dir}")

# Optional: Push to Hugging Face Hub
# model.push_to_hub(
#     "your_username/gpt_oss_alpaca_lora",
#     token="YOUR_HF_TOKEN"
# )
# tokenizer.push_to_hub(
#     "your_username/gpt_oss_alpaca_lora",
#     token="YOUR_HF_TOKEN"
# )

## Step 9: Load and Use Saved Model (Optional)

Load the fine-tuned model in a new session.

In [ ]:
# Uncomment to load saved model
# from unsloth import FastLanguageModel
# 
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="gpt_oss_lora_adapters",  # Path to saved adapters
#     max_seq_length=1024,
#     dtype=None,
#     load_in_4bit=True,
# )
# 
# print("Model loaded successfully!")

## Step 10: Export for Deployment (Optional)

Export the model in different formats for production use:
- **merged_16bit**: Full precision for vLLM or other inference engines
- **mxfp4**: 4-bit quantized format for efficient deployment

In [ ]:
# Export as 16-bit merged model
# model.save_pretrained_merged(
#     "gpt_oss_alpaca_16bit",
#     tokenizer,
#     save_method="merged_16bit"
# )

# Export as mxfp4 4-bit model
# model.save_pretrained_merged(
#     "gpt_oss_alpaca_4bit",
#     tokenizer,
#     save_method="mxfp4"
# )

# Push to Hugging Face Hub
# model.push_to_hub_merged(
#     "your_username/gpt_oss_alpaca_16bit",
#     tokenizer,
#     save_method="merged_16bit",
#     token="YOUR_HF_TOKEN"
# )

print("Export options available - uncomment code blocks to use")

## Summary

This notebook demonstrated:
1. Loading GPT-OSS 20B with 4-bit quantization
2. Configuring LoRA adapters for efficient fine-tuning
3. Preparing the Alpaca instruction dataset (52K examples)
4. Training with response-only masking
5. Testing the fine-tuned model
6. Saving and exporting the model

### Next Steps
- Increase `max_steps` or use `num_train_epochs=1` for full training on all 52K examples
- Experiment with different LoRA ranks (r=16, 32, 64) for better performance
- Try different reasoning effort levels during inference
- Use alternative datasets:
  - `databricks/databricks-dolly-15k` for high-quality examples
  - `Open-Orca/OpenOrca` for larger scale training
  - Your own custom dataset for domain-specific tasks
- Deploy using vLLM or other inference frameworks
- Fine-tune on task-specific data for specialized applications

### Dataset Format for Custom Data
To use your own dataset, format it like this:
```python
{
    "instruction": "What is the capital of France?",
    "input": "",  # Optional context or additional information
    "output": "The capital of France is Paris."
}
```

Or use the messages format directly:
```python
{
    "messages": [
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."}
    ]
}
```

### Resources
- [Unsloth Documentation](https://unsloth.ai/docs/)
- [GPT-OSS Cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [Alpaca Dataset](https://github.com/tatsu-lab/stanford_alpaca)
- [Hugging Face Datasets](https://huggingface.co/datasets)